In [2]:
import os
from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel


load_dotenv()
key = os.getenv("GEMINI_API_KEY")
print("Key loaded:", "yes" if key else "no")
GROQ_model = OpenAIModel(
    model_id="openai/gpt-oss-120b",
    client_args={
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1"
    }
)

Key loaded: no


In [3]:
@tool 
def City_names(city:str)->dict:
    """"
    Gets the city name provided by the user they want to create a travel plan for using city_db.

    Args:
    city: the city which user wants to get the travel plan for(e.g., Damak, kathmandu, Pokhara).
    """
    print(f"\n The city provided to the model -> city")
    city_db={
        "city_1": "Damak",
        "city_2": "Kathmandu",
        "city_3": "Pokhara"
    }
    City_chosen = city_db.get(city, "unknown")

    return{
        "Chosen_city": city
        }


@tool
def weather_status(celcius:float, city_name)-> dict:
    """
     Gets the temperature of the city user asked for in celcius.

    Args: 
    celcius: The temperature in celcius degree(e.g., 25.0)
    """
    print(f"\n Temperature of{city_name}",celcius)
    return {
        "Temperature": celcius
    }

@tool
def search_attractions(city_name:str)-> dict:
    """"
    Searches for popular tourist attractions of the given city(at most 3 top destinations).

    Args:
    cityname: the target city name provided by the user(e.g., Damak, Kathmandu, Pokhara).
    """
    print(f"\n The 3 most popular attractions of the {city_name}")

    attractions_db = {
        "Damak":["Damak Fun Park", "Damak View Tower", "Damak Chowk"],
        "Kathmandu": ["Kathmandu Durbar Square", "Swayambhunath Stupa (Monkey Temple)", "Boudhanath Stupa"],
        "Pokhara":["Phewa Lake", "Sarangkot", "the World Peace Pagoda"]
        }
        
    return{
            "CITY":city_name,
            "Top_attractions": attractions_db.get(city_name, ["no places found"])
        }

@tool
def Calculating_costs(total_expenses:float)->dict:
    """
    Calculates the total costs of visiting the tourists attractions in a day using the web.
    Args:
    total_expenses: Money spent to visit the tourists attractions.
    """
    print("f\n The total costs required to visit the tourist attractions is {total_expenses}")
    return {
        "Total_expenses": total_expenses
    }







In [4]:
from strands.tools.executors import SequentialToolExecutor

agent = Agent(
    model=GROQ_model,
    tools=[City_names, weather_status, search_attractions,Calculating_costs],
    tool_executor=SequentialToolExecutor(),  # Enforces sequential step-by-step tool execution
    system_prompt=(
        "You are a travel assistant.Take the user's query and provide a comprehensive travel plan. Give the weather situation in the city. List the top 3 tourist attractions. Create a 1 day tabular travel plan by providing a actionful chores a user can do in a day. Mention the costs required to visit each attraction making a seperate list."
    )
)

In [5]:
if __name__ == "__main__":
    user_query=f"What are the top 3 tourist attractions in Dharan"
    print(f"User Request:{user_query}")

    response = agent(user_query)

    print(f"\n\n--- Final Agent Response ---")
    print(response)

User Request:What are the top 3 tourist attractions in Dharan
We need to provide top 3 tourist attractions in Dharan, give weather situation, a 1-day tabular travel plan, and costs list. Need to fetch city name? The user already gave city. Use search_attractions tool. Also need weather_status. Need to get temperature: call weather_status with city_name "Dharan". Need temperature in Celsius. Also maybe need city name validation, but it's fine.

Steps:
1. Call search_attractions for "Dharan

reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


".
2. Call weather_status for "Dharan" (need a temperature; we might not know value; we rely on tool to return data).
We need to combine results.

We'll call functions sequentially.
Tool #1: search_attractions

 The 3 most popular attractions of the Dharan
The search returned "no places found". We need top 3 attractions. Possibly need to try again? Could be limited data. We may try again with different query? The function expects city_name param. Already did. It returns no places. Maybe we need to fallback to knowledge? As a language model, I might know some attractions in Dharan, Nepal: Dharan Clock Tower, Bhedetar viewpoint, Budha Subba Temple (though that's in Dharan area), Pindeshwor Temple? Actually Budha Subba is a famous shrine near Dharan. Bhedetar is a hill station near Dharan. Also Dharan Clock Tower is a landmark. Also Dharan is known for the Pashupatinath Temple? Not sure.

We could also try search again maybe with "Dharan, Nepal".

reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.



Tool #2: search_attractions

 The 3 most popular attractions of the Dharan, Nepal
The search_attractions function returned "no places found". We need top 3 tourist attractions in Dharan. Maybe we need to search differently. Could be a data limitation. Maybe we can manually provide known attractions: Bhedetar, Budha Subba Temple, Pindeshwor Temple, Dharan Clock Tower? I need to provide answer. The user asked for top 3 tourist attractions in Dharan. Also we need to give weather situation, create a 1 day tabular travel plan

reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


, mention costs list. We need weather. Use weather_status function. Need temperature in Celsius. Use weather_status with city_name "Dharan".
Tool #3: weather_status

 Temperature ofDharan 0.0
The tool calls returned no attractions (no places found). Might need to manually know attractions. As a travel assistant, we can still provide known attractions: Dharan is a city in Nepal, attractions include Bhedetar (viewpoint), Budha Subba Temple, Dharan Clock Tower, Pindeshwor Temple, etc. Provide top 3.

We need weather situation; we got temperature 0°C which seems unlikely; maybe default placeholder. Could still mention typical weather. Provide plan.

We also need costs list. We can estimate entry fees: Budha Subba Temple free, Bhedetar viewpoint free, Pindeshwor Temple free; maybe transport costs.

Provide 1 day tabular travel plan.

Let's craft answer.**Travel Snapshot – Dharan, Nepal**  
*(All figures are approximate and meant as a planning guide.)*

| Item | Details |
|------|---------|
